# Exploration des sources

Avant d'écrire les scripts d'extraction, je teste ce que renvoie chaque source retenue à l'étape 1 :

- la structure de la réponse (JSON, XML ou HTML)
- les champs disponibles pour le texte, l'image et la date
- si les liens d'images fonctionnent vraiment

In [1]:
import os
from urllib.robotparser import RobotFileParser

import feedparser
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

load_dotenv("../.env")
API_KEY = os.getenv("NEWSDATA_API_KEY")
HEADERS = {"User-Agent": "Mozilla/5.0 (projet etudiant multimodal-news-extraction)"}

## 1. NewsData.io

In [2]:
# Un appel = une page de 10 articles = 1 crédit sur les 200 du jour
response = requests.get(
    "https://newsdata.io/api/1/latest",
    params={"language": "fr", "country": "fr"},
    headers={"X-ACCESS-KEY": API_KEY},
    timeout=15,
)
data = response.json()
print("Code HTTP :", response.status_code)
print("Clés de la réponse :", list(data.keys()))
print("Articles sur la page :", len(data["results"]))

Code HTTP : 200
Clés de la réponse : ['status', 'totalResults', 'results', 'nextPage']
Articles sur la page : 10


In [3]:
article = data["results"][0]
for key, value in article.items():
    print(f"{key:18} {str(value)[:70]}")

article_id         8316ad1821ce7a0aaf07b51509520404
link               https://www.france24.com/fr/france/20260927-ultime-journ%C3%A9e-crucia
title              Ultime journée cruciale au procès de Rachida Dati et Carlos Ghosn
description        La dernière journée d'audience du procès pour corruption de l'ancienne
content            ONLY AVAILABLE IN PAID PLANS
keywords           ['france']
creator            ['france 24']
language           french
country            ['france']
category           ['world']
datatype           news
pubDate            2026-09-27 23:04:10
pubDateTZ          UTC
fetched_at         2026-09-28 07:01:06
image_url          https://s.france24.com/media/display/bf48ed80-bac7-11f1-8635-555a0aff8
video_url          None
source_id          france24_fr
source_name        France 24 French
source_priority    3118
source_url         https://www.france24.com
source_icon        https://i.newsdata.io/france24_fr.png
sentiment          ONLY AVAILABLE IN PROFESSIONAL AND CO

In [4]:
results = data["results"]
print("Articles avec image_url :", sum(1 for a in results if a["image_url"]), "/", len(results))
print("Doublons signalés :", sum(1 for a in results if a["duplicate"]))
print("Exemples de source_priority :", [a["source_priority"] for a in results[:5]])

Articles avec image_url : 6 / 10
Doublons signalés : 5
Exemples de source_priority : [3118, 6688, 1375, 611, 1375]


In [5]:
# Test d'un lien d'image : il doit répondre 200 et renvoyer un type image/...
image_url = next(a["image_url"] for a in results if a["image_url"])
r = requests.get(image_url, headers=HEADERS, timeout=15)
print(r.status_code, r.headers.get("Content-Type"), len(r.content), "octets")

200 image/jpeg 88048 octets


Observations :

- le champ `content` contient seulement "ONLY AVAILABLE IN PAID PLANS", je prends donc `description` comme texte
- certains articles n'ont pas d'`image_url` : on les garde, avec l'image vide
- `pubDate` est en UTC au format `AAAA-MM-JJ HH:MM:SS`
- `creator` est une liste (plusieurs auteurs possibles)
- le champ `duplicate` permet d'ignorer les articles déjà repris d'une autre source
- la clé API peut passer dans l'en-tête `X-ACCESS-KEY`, ce qui évite qu'elle apparaisse dans l'URL

## 2. Flux RSS

In [6]:
FEEDS = {
    "lemonde": "https://www.lemonde.fr/rss/une.xml",
    "franceinfo": "https://www.franceinfo.fr/titres.rss",
    "lesdecodeurs": "https://www.lemonde.fr/les-decodeurs/rss_full.xml",
    "20minutes_fakeoff": "https://www.20minutes.fr/feeds/rss-fake-off.xml",
}

feeds = {}
for name, url in FEEDS.items():
    r = requests.get(url, headers=HEADERS, timeout=15)
    feeds[name] = feedparser.parse(r.content)
    entries = feeds[name].entries
    with_image = sum(1 for e in entries if e.get("media_content") or e.get("enclosures"))
    print(f"{name:18} {len(entries)} articles, {with_image} avec image")

lemonde            16 articles, 16 avec image


franceinfo         28 articles, 28 avec image


lesdecodeurs       20 articles, 20 avec image


20minutes_fakeoff  20 articles, 20 avec image


In [7]:
# Où se trouve l'image selon le média ?
for name, feed in feeds.items():
    entry = feed.entries[0]
    print(name)
    print("   media_content :", entry.get("media_content"))
    print("   enclosures    :", entry.get("enclosures"))

lemonde
   media_content : [{'width': '644', 'height': '322', 'url': 'https://img.lemde.fr/2026/09/28/240/0/1600/800/644/322/60/0/dd2bccc_upload-1-9q4ykgkknzpk-whatsapp-image-2026-09-28-at-12-07-07.jpeg'}]
   enclosures    : []
franceinfo
   media_content : None
   enclosures    : [{'length': '243', 'type': 'image/jpeg', 'href': 'https://www.franceinfo.fr/pictures/Kk5Auhu3xsbXPNLmslLpAIFEm-0/0x0:2000x1125/432x243/filters:format(jpg)/2026/09/28/6aba364ad5302173657480.jpg'}]
lesdecodeurs
   media_content : [{'width': '644', 'height': '322', 'url': 'https://img.lemde.fr/2026/03/30/236/0/2831/1415/644/322/60/0/ff58abc_ftp-1-wwtasvshaq0y-1ed6a0f0ed8e4814abd23dd2b6a63ca9-0-7ba059f7fc3046419834791cafe9e89f.jpg'}]
   enclosures    : []
20minutes_fakeoff
   media_content : None
   enclosures    : [{'type': 'image/jpeg', 'length': '0', 'href': 'https://img.20mn.fr/7rAkGvePTsaR8JQc1zV4rSk/1200x768_8-septembre-2026-bagdad-bagdad-chine-bagdad-irak-a-dans-le-contexte-de-l-expansion-rapide-des-platef

In [8]:
entry = feeds["lemonde"].entries[0]
print("Titre :", entry.title)
print("Résumé :", entry.summary[:150])
print("Date :", entry.published, "->", entry.published_parsed)

Titre : EN DIRECT, voyage du pape à Metz : le pape Léon XIV déplore le retour de la guerre en Europe, un « massacre inutile [qui] fait chaque jour d’innombrables victimes civiles »
Résumé : Léon XIV effectue la dernière étape de son voyage en France avec une visite à Metz, où il a prononcé lundi midi un discours sur l’Europe et la paix in
Date : Mon, 28 Sep 2026 11:38:34 +0200 -> time.struct_time(tm_year=2026, tm_mon=9, tm_mday=28, tm_hour=9, tm_min=38, tm_sec=34, tm_wday=0, tm_yday=271, tm_isdst=0)


Observations :

- tous les articles des 4 flux ont une image
- Le Monde et Les Décodeurs mettent l'image dans `media_content`, franceinfo et 20 Minutes dans `enclosures`
- le texte se limite au résumé (`summary`)
- feedparser convertit déjà la date en UTC dans `published_parsed`

## 3. Le Gorafi (scraping)

In [9]:
# Le flux RSS du Gorafi ne contient pas d'image, d'où le scraping
gorafi_feed = feedparser.parse(requests.get("https://www.legorafi.fr/feed/", headers=HEADERS, timeout=15).content)
print("Articles avec image dans le flux :",
      sum(1 for e in gorafi_feed.entries if e.get("media_content") or e.get("enclosures")),
      "/", len(gorafi_feed.entries))

Articles avec image dans le flux : 0 / 20


In [10]:
# Vérification du robots.txt avant de scraper
robots_txt = requests.get("https://www.legorafi.fr/robots.txt", headers=HEADERS, timeout=15).text
print(robots_txt)

robots = RobotFileParser()
robots.parse(robots_txt.splitlines())


User-agent: *
Allow: /wp-content/uploads/
Disallow: /wp-content/plugins/
Disallow: /wp-admin/
Disallow: /wp-includes/
Disallow: /wp-login.php
Disallow: /readme.html
Sitemap: https://www.legorafi.fr/sitemap_index.xml
Sitemap: https://www.legorafi.fr/news-sitemap.xml



In [11]:
home = BeautifulSoup(requests.get("https://www.legorafi.fr/", headers=HEADERS, timeout=15).content, "html.parser")

links = []
for a in home.find_all("a", href=True):
    if a["href"].startswith("https://www.legorafi.fr/20") and a["href"] not in links:
        links.append(a["href"])

print(len(links), "liens d'articles sur la page d'accueil")
print(links[0])
print("Autorisé par robots.txt :", robots.can_fetch(HEADERS["User-Agent"], links[0]))

48

 liens d'articles sur la page d'accueil
https://www.legorafi.fr/2026/09/24/rachida-dati-parvient-a-sechapper-de-son-proces-dans-un-coffre-a-bijoux/
Autorisé par robots.txt : True


In [12]:
page = BeautifulSoup(requests.get(links[0], headers=HEADERS, timeout=15).content, "html.parser")

print("Titre :", page.find("h1").get_text(strip=True))
print("Image :", page.find("meta", property="og:image")["content"])
print("Date :", page.find("meta", property="article:published_time")["content"])
print("Auteur :", page.find("meta", attrs={"name": "author"})["content"])
print()
for p in page.find("div", id="mvp-content-main").find_all("p"):
    print("-", p.get_text(strip=True)[:100])

Titre : Rachida Dati parvient à s’échapper de son procès dans un coffre à bijoux
Image : https://www.legorafi.fr/wp-content/uploads/2026/02/GettyImages-2251552038-1000x600.jpg
Date : 2026-09-24T12:20:36+00:00
Auteur : La Rédaction

- Alors que son procès ne devait s’achever que le 28 septembre, Rachida Dati y a mis un terme en s’éch
- Une évasion discrète
- Accusée d’avoir réalisé illégalement un travail de lobbying au Parlement européen pour Renault-Nissa
- Pour le moment introuvable, l’ex-garde des Sceaux a toutefois publié un message sur X ce matin :« Pl
- 
- Rejoignez le club des Gens qui Lisent Encore des Mails avec la newsletter du Gorafi
- Δ
- Enfin des arbres abattus pour une bonne raison !


Observations :

- le robots.txt autorise les pages d'articles (seuls les dossiers techniques de WordPress sont interdits)
- les URL d'articles commencent toutes par `https://www.legorafi.fr/20..`, ce qui permet de les repérer sur la page d'accueil
- l'image principale est dans la balise `og:image` et la date dans `article:published_time`
- le texte est dans les `<p>` de la div `mvp-content-main`. Il y a des paragraphes vides à retirer, et tout ce qui suit l'encart newsletter ne fait pas partie de l'article

## Conclusion

Les trois types de sources sont exploitables. Pour les scripts :

- un module par source, qui renvoie des articles au même format
- les articles sans texte sont écartés, ceux sans image sont gardés
- quand il y a une image, elle est téléchargée pour vérifier que le lien fonctionne et que c'est bien une image (`Content-Type`)
- une pause entre les pages du Gorafi et un nombre de pages limité pour NewsData